In [ ]:
import sys
import os
from google.colab import drive

drive.mount('/content/drive')

dossier_projet = '/content/drive/MyDrive/Colab Notebooks/projet_7'

os.chdir(dossier_projet)

sys.path.append(dossier_projet)

print("Dossier actuel :", os.getcwd())
print("Contenu du dossier :", os.listdir())

Mounted at /content/drive
Dossier actuel : /content/drive/MyDrive/Colab Notebooks/projet_7
Contenu du dossier : ['app.py', 'utils.py', 'data', 'projet_7.ipynb']


In [ ]:
!pip install pypdf

In [ ]:
from utils import (afficher_hotels, afficher_voyage, afficher_vols, charger_brochures,
                   charger_encodeur, connexion, encoder, requete, texte_proposition)

conn = connexion()
brochures = charger_brochures()
encodeur = charger_encodeur()

print(f"{len(brochures)} brochures d'hôtels chargées")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

135 brochures d'hôtels chargées


Partie 1 - Le chargement des données

In [ ]:
print("=== Les tables de la base de données ===")
display(requete(conn, "SELECT name FROM sqlite_master WHERE type='table'"))

=== Les tables de la base de données ===


,name
0,sqlite_sequence
1,vols
2,activites
3,reservations


In [ ]:
print("=== Un aperçu de la table `vols` ===")
display(requete(conn, "SELECT * FROM vols LIMIT 3"))

=== Un aperçu de la table `vols` ===


,id,numero,origine,destination,date_depart,heure_depart,duree_h,prix_eur,places_restantes
0,1,U22681,Nice,Madrid,2026-07-01,11:30,2.2,92.0,81
1,2,LH9975,Nice,Madrid,2026-07-01,20:20,2.2,111.0,77
2,3,AF2322,Nice,Madrid,2026-07-01,07:40,1.9,102.0,14


In [ ]:
print("=== Un aperçu de la table `activités` ===")
display(requete(conn, "SELECT * FROM activites LIMIT 3"))
print("=== Et un exemple des activités de Barcelone ===")
display(requete(conn, "SELECT * FROM activites WHERE ville = 'Barcelone'"))

=== Un aperçu de la table `activités` ===


,id,ville,nom,categorie,duree_h,prix_eur
0,1,Madrid,Musée du Prado,culture,3.0,15.0
1,2,Madrid,Palais royal de Madrid,culture,2.0,14.0
2,3,Madrid,Stade Santiago Bernabéu,culture,1.5,35.0


=== Et un exemple des activités de Barcelone ===


,id,ville,nom,categorie,duree_h,prix_eur
0,9,Barcelone,Sagrada Familia,culture,2.0,26.0
1,10,Barcelone,Casa Batlló,culture,1.5,29.0
2,11,Barcelone,Musée Picasso,culture,2.0,15.0
3,12,Barcelone,Parc Güell,plein air,2.0,18.0
4,13,Barcelone,Plage de la Barceloneta,plein air,3.0,0.0
5,14,Barcelone,Marché de la Boqueria,gastronomie,1.5,0.0
6,15,Barcelone,Dégustation de tapas dans le quartier d'El Born,gastronomie,3.0,45.0
7,16,Barcelone,Spectacle de la Fontaine magique de Montjuïc,nocturne,1.0,0.0


In [ ]:
print("=== Les brochures d'hôtels (des PDF, comme au Projet 04) ===")
display(brochures[["ville", "hotel", "etoiles", "note", "avis", "prix_nuit"]].head(12))
print(f"({len(brochures)} brochures en tout, {brochures['ville'].nunique()} villes)")

=== Les brochures d'hôtels (des PDF, comme au Projet 04) ===


,ville,hotel,etoiles,note,avis,prix_nuit
0,Amsterdam,Auberge Helios,1,6.2,805,80.0
1,Amsterdam,B&B Wanderlust,5,8.6,793,283.0
2,Amsterdam,Hôtel Bocca,3,7.8,1909,169.0
3,Amsterdam,Hôtel Florentina,2,7.4,1885,148.0
4,Amsterdam,Hôtel Jardins,3,7.5,551,155.0
5,Amsterdam,Hôtel Tramontana,3,7.8,1359,158.0
6,Amsterdam,Résidence Aurora,4,8.1,497,178.0
7,Amsterdam,Résidence Corso,3,7.5,490,172.0
8,Amsterdam,Résidence Doria,3,8.3,821,194.0
9,Athenes,Auberge Ibiza,3,8.1,2176,120.0


(135 brochures en tout, 15 villes)


In [ ]:
print(brochures[brochures["ville"] == "Madrid"].iloc[0]["texte"])

Auberge Jardins Madrid - 4 etoiles - a partir de 182 euros la nuit Note des voyageurs : 8.1 sur 10 (697 avis) Presentation Un havre de calme dans une rue sans passage, pensé pour se reposer vraiment. L'établissement se trouve au coeur de la vieille ville, dans une ruelle pavée. Il dispose de une piscine extérieure chauffée, ouverte de mai à octobre. Equipements wifi gratuit, piscine exterieure, salle de sport, restaurant, climatisation, animaux acceptés. Avis des clients "Un hôtel très calme, on a dormi comme des bébés." "On ne s'attendait pas à une si belle piscine." "L'ascenseur est minuscule, prévoyez deux voyages." Auberge Jardins - Madrid - Brochure du reseau d'hoteliers de Marc


Partie 2 - Le premier outil : Le sélectionneur de vols d'avion

Étape 1 : chercher les vols

In [ ]:
def outil_vols(conn, destination, date_depart, voyageurs=1):
    """
    Outil : trouve les vols disponibles vers une destination, un jour donné.

    Arguments :
    conn -- une connexion ouverte vers voyages.db
    destination -- le nom de la ville, tel qu'écrit dans la table
    date_depart -- le jour du départ, au format "2026-08-12"
    voyageurs -- le nombre de places nécessaires

    Retourne :
    vols -- un DataFrame des vols trouvés, du moins cher au plus cher
    """
    sql = """
        SELECT numero, origine, heure_depart, duree_h, prix_eur, places_restantes
        FROM vols
        WHERE destination = ? AND date_depart = ? AND places_restantes >= ?
        ORDER BY prix_eur ASC
    """  # (1) les colonnes, (2) les filtres, (3) les places, (4) le tri
    return requete(conn, sql, (destination, date_depart, voyageurs))


vols_madrid = outil_vols(conn, "Madrid", "2026-08-12", voyageurs=2)
print(f"{len(vols_madrid)} vol(s) pour Madrid le 12 août :")
afficher_vols(vols_madrid)

assert not vols_madrid.empty, "On attend au moins un vol pour Madrid ce jour-là"
assert list(vols_madrid.columns) == ["numero", "origine", "heure_depart", "duree_h",
                                      "prix_eur", "places_restantes"], \
    "Les colonnes attendues sont numero, origine, heure_depart, duree_h, prix_eur, places_restantes"
assert vols_madrid["prix_eur"].is_monotonic_increasing, "Les vols doivent être triés du moins cher au plus cher"
assert (vols_madrid["places_restantes"] >= 2).all(), "Il faut au moins 2 places dans chaque vol retourné"
print("\nExercice validé !")

7 vol(s) pour Madrid le 12 août :
  W65397   Paris CDG  -> 22:05  2.0 h     124 EUR  (38 places)
  VY8043   Nice       -> 06:15  2.2 h     130 EUR  (35 places)
  LH2158   Nice       -> 07:40  1.9 h     132 EUR  (80 places)
  AF2826   Paris CDG  -> 11:30  2.2 h     183 EUR  (10 places)
  FR7730   Nice       -> 06:15  2.2 h     205 EUR  (75 places)

Exercice validé !


In [ ]:
def outil_activites(conn, ville):
    """
    Outil : liste les activités proposées dans une ville, de la moins chère à la plus chère.

    Arguments :
    conn -- une connexion ouverte vers voyages.db
    ville -- le nom de la ville

    Retourne :
    activites -- un DataFrame des activités de la ville
    """
    # (1) les colonnes, (2) le filtre sur la ville, (3) le tri de la moins chère à la plus chère
    sql = "SELECT nom, categorie, duree_h, prix_eur FROM activites WHERE ville = ? ORDER BY prix_eur ASC"
    return requete(conn, sql, (ville,))


activites_rome = outil_activites(conn, "Rome")
display(activites_rome)

attendu = requete(conn, "SELECT COUNT(*) AS n FROM activites WHERE ville = 'Rome'").iloc[0]["n"]

assert list(activites_rome.columns) == ["nom", "categorie", "duree_h", "prix_eur"], \
    "Les colonnes attendues sont nom, categorie, duree_h et prix_eur, dans cet ordre"
assert len(activites_rome) == attendu, \
    f"On attend les {attendu} activités de Rome, tu en retournes {len(activites_rome)} : " \
    "as-tu bien filtré sur la ville ?"
assert activites_rome["prix_eur"].is_monotonic_increasing, \
    "Les activités doivent être triées de la moins chère à la plus chère"
print("\nExercice validé !")

,nom,categorie,duree_h,prix_eur
0,Fontaine de Trevi,plein air,1.0,0.0
1,Via Appia Antica,plein air,3.0,0.0
2,Marché de Testaccio,gastronomie,1.5,0.0
3,Panthéon de Rome,culture,1.0,5.0
4,Glacier Giolitti,gastronomie,1.0,5.0
5,Colisée,culture,2.5,18.0
6,Musées du Vatican et Chapelle Sixtine,culture,3.0,25.0
7,Opéra en plein air aux Thermes de Caracalla,nocturne,3.0,45.0



Exercice validé !


Partie 3 - L'outil des hôtels

In [ ]:
print("Les colonnes du tableau brochures :", list(brochures.columns))

exemple = brochures[brochures["ville"] == "Madrid"].iloc[0]
print(f"\nNotre hôtel d'exemple : {exemple['hotel']}, {exemple['etoiles']} étoiles, note {exemple['note']}/10")

print("\n=== Sa colonne resume : la présentation et les avis clients, rien d'autre ===")
print(exemple["resume"])

Les colonnes du tableau brochures : ['ville', 'hotel', 'etoiles', 'note', 'avis', 'prix_nuit', 'texte', 'resume']

Notre hôtel d'exemple : Auberge Jardins, 4 étoiles, note 8.1/10

=== Sa colonne resume : la présentation et les avis clients, rien d'autre ===
Un havre de calme dans une rue sans passage, pensé pour se reposer vraiment. L'établissement se trouve au coeur de la vieille ville, dans une ruelle pavée. Il dispose de une piscine extérieure chauffée, ouverte de mai à octobre. "Un hôtel très calme, on a dormi comme des bébés." "On ne s'attendait pas à une si belle piscine." "L'ascenseur est minuscule, prévoyez deux voyages."


In [ ]:
def outil_hotels(brochures, encodeur, ville, envie, k=3):
    """
    Outil : trouve les hôtels d'une ville qui correspondent le mieux à une envie écrite librement.

    Arguments :
    brochures -- le DataFrame retourné par charger_brochures
    encodeur -- le modèle retourné par charger_encodeur
    ville -- le nom de la ville
    envie -- ce que cherche le voyageur, avec ses mots à lui
    k -- le nombre d'hôtels à retourner

    Retourne :
    hotels -- un DataFrame des k meilleures brochures, avec une colonne "score", les meilleures d'abord
    """
    hotels_ville = brochures[brochures["ville"] == ville].reset_index(drop=True)

    vecteurs = encoder(encodeur, hotels_ville["resume"]) # (1) une ligne par brochure de la ville
    vecteur_envie = encoder(encodeur, [envie])[0] # (2) un seul vecteur pour la demande
    hotels_ville["score"] = hotels_ville["score"] = vecteurs @ vecteur_envie # (3) vecteurs normalisés : produit scalaire = cosinus
    hotels = (hotels_ville.sort_values("score", ascending=False).head(k).reset_index(drop=True)) # (4)
    return hotels


ENVIE = "un hôtel pour la famille avec une piscine, et des visites dans la ville"

for envie in [ENVIE, "je veux faire la fête et sortir le soir"]:
    print(f'« {envie} »')
    afficher_hotels(outil_hotels(brochures, encodeur, "Madrid", envie))
    print()

famille = outil_hotels(brochures, encodeur, "Madrid", "en famille avec des enfants")
piscine = outil_hotels(brochures, encodeur, "Madrid", "une piscine pour nager")

assert "score" in famille.columns, "Il manque la colonne score"
assert len(famille) == 3, "On attend k = 3 hotels"
assert famille["score"].is_monotonic_decreasing, "Les hotels doivent etre tries du plus au moins pertinent"
assert any(mot in famille.loc[0, "texte"].lower() for mot in ("famille", "enfants")), \
    f"Pour la famille on attend un hotel familial, tu obtiens {famille.loc[0, 'hotel']}"
assert "piscine" in piscine.loc[0, "texte"].lower(), \
    f"Pour une piscine on attend un hotel avec piscine, tu obtiens {piscine.loc[0, 'hotel']}"
print("Exercice validé ! Le même modèle, deux demandes, deux hôtels opposés.")

« un hôtel pour la famille avec une piscine, et des visites dans la ville »
  0.788  Hôtel Corso            3*    138 EUR/nuit  note 8.2/10 (660 avis)
          Un hôtel familial pensé pour les familles avec enfants, avec des chambres communicantes. L'établissement se trouve à quinze minutes du centre en ...
  0.673  Maison Riviera         5*    214 EUR/nuit  note 8.8/10 (615 avis)
          Un palace de grand luxe au service irréprochable : marbre, conciergerie, voiturier à l'entrée. L'établissement se trouve sur les quais, avec une ...
  0.664  Hôtel Esperanza        4*    187 EUR/nuit  note 7.7/10 (1467 avis)
          Un hôtel design, entre galerie d'art et espace de coworking, très photogénique. L'établissement se trouve sur les quais, avec une vue dégagée sur...

« je veux faire la fête et sortir le soir »
  0.510  Maison Ximena          3*    150 EUR/nuit  note 7.8/10 (2192 avis)
          Un hôtel très animé, en plein coeur du quartier qui fait la fête, entouré de bars ouverts 

Partie 4 - Le cerveau de l'agent : décider

In [ ]:
from datetime import datetime, timedelta


def chiffrer_voyage(vol, hotel, activites, nuits):
    """
    Calcule le prix total d'un voyage, par personne.

    Arguments :
    vol -- une ligne du DataFrame des vols
    hotel -- une ligne du DataFrame des hôtels
    activites -- une liste de dictionnaires, chacun avec une clé "prix_eur"
    nuits -- le nombre de nuits

    Retourne :
    total -- le prix total en euros, par personne
    """
    prix_vol = vol["prix_eur"]
    prix_hotel = hotel["prix_nuit"] * nuits
    prix_activites = sum(a["prix_eur"] for a in activites)
    return float(prix_vol + prix_hotel + prix_activites)


def dates_voisines(date_depart, ecart=2):
    """
    Les jours à explorer autour de la date demandée, du plus proche au plus lointain.

    Arguments :
    date_depart -- le jour souhaité, au format "2026-08-12"
    ecart -- de combien de jours on accepte de s'éloigner

    Retourne :
    voisines -- la liste des autres dates, sans la date demandée elle-même
    """
    jour = datetime.strptime(date_depart, "%Y-%m-%d")
    voisines = []
    for decalage in range(1, ecart + 1):
        for signe in (-1, 1):
            voisines.append((jour + timedelta(days=signe * decalage)).strftime("%Y-%m-%d"))
    return voisines


vol_test = vols_madrid.iloc[0]
hotel_test = outil_hotels(brochures, encodeur, "Madrid", ENVIE).iloc[0]
activites_test = outil_activites(conn, "Madrid").to_dict("records")

print(f"Vol {vol_test['numero']} : {vol_test['prix_eur']:.0f} EUR")
print(f"{hotel_test['hotel']} x 4 nuits : {hotel_test['prix_nuit'] * 4:.0f} EUR")
print(f"{len(activites_test)} activités : {sum(a['prix_eur'] for a in activites_test):.0f} EUR")
print(f"TOTAL : {chiffrer_voyage(vol_test, hotel_test, activites_test, 4):.0f} EUR")
print(f"\nAutour du 12 août, l'agent pourra aussi regarder : {dates_voisines('2026-08-12')}")

Vol W65397 : 124 EUR
Hôtel Corso x 4 nuits : 552 EUR
8 activités : 130 EUR
TOTAL : 806 EUR

Autour du 12 août, l'agent pourra aussi regarder : ['2026-08-11', '2026-08-13', '2026-08-10', '2026-08-14']


In [ ]:
def essayer_une_date(demande, conn, brochures, encodeur):
    """
    Compose le meilleur voyage possible pour UN jour de départ donné.

    Arguments :
    demande -- un dictionnaire avec destination, date_depart, nuits, voyageurs, budget_max, envie
    conn, brochures, encodeur -- les sources de données de l'agent

    Retourne :
    voyage -- un dictionnaire décrivant le voyage, ou None si rien ne rentre dans le budget
    journal -- la liste des ajustements faits par l'agent, une phrase par sacrifice
    """
    vols = outil_vols(conn, demande["destination"], demande["date_depart"], demande["voyageurs"])
    hotels = outil_hotels(brochures, encodeur, demande["destination"], demande["envie"])
    activites = outil_activites(conn, demande["destination"])

    journal = []
    if vols.empty or hotels.empty:
        return None, ["aucun vol ou aucun hôtel disponible pour cette destination et cette date"]

    vol = vols.iloc[0]                              # le vol le moins cher
    retenues = activites.to_dict("records")         # toutes les activités, pour commencer
    impossible = False

    # L'ordre dans lequel on essaiera les hôtels : le plus pertinent d'abord, puis ceux qui
    # coûtent vraiment moins cher que lui, du plus cher au moins cher, pour descendre en douceur.
    prix_du_premier = hotels.iloc[0]["prix_nuit"]
    replis = [j for j in range(1, len(hotels)) if hotels.iloc[j]["prix_nuit"] < prix_du_premier]
    ordre = [0] + sorted(replis, key=lambda j: hotels.iloc[j]["prix_nuit"], reverse=True)
    position = 0                                    # où on en est dans la liste ordre

    for _ in range(30): # garde-fou anti-boucle infinie
        hotel = hotels.iloc[ordre[position]]
        total = chiffrer_voyage(vol, hotel, retenues, demande["nuits"])

        # Une ligne par combinaison essayée, pour voir l'agent chercher.
        ecart = total - demande["budget_max"]
        verdict = "OK" if ecart <= 0 else f"dépasse de {ecart:.0f}"
        print(f"   {demande['date_depart'][8:]}/{demande['date_depart'][5:7]}  {hotel['hotel'][:20]:20s} "
              f"vol {vol['prix_eur']:4.0f} + hôtel {hotel['prix_nuit'] * demande['nuits']:5.0f} "
              f"+ sorties {sum(a['prix_eur'] for a in retenues):4.0f} = {total:6.0f} EUR   {verdict}")

        if total <= demande["budget_max"]:
            break                                   # le plan tient dans le budget : on s'arrête là

        payantes = [a for a in retenues if a["prix_eur"] > 0]

        if payantes: # repli n°1 : sacrifier une sortie
            plus_chere = max(payantes, key=lambda a: a["prix_eur"]) # (1) la sortie la plus chère
            retenues = [a for a in retenues if a is not plus_chere] # (2) on la retire du programme
            position = 0 # (3) on repart du meilleur hôtel
            journal.append(f"j'ai retiré « {plus_chere['nom']} » ({plus_chere['prix_eur']:.0f} EUR)")

        elif position + 1 < len(ordre):             # repli n°2 : descendre d'un hôtel
            position += 1 # (4) l'hôtel suivant de la liste
            journal.append(f"{hotel['hotel']} restait trop cher, "
                           f"j'ai pris {hotels.iloc[ordre[position]]['hotel']} à la place")

        else: # plus rien à relâcher
            impossible = True # (5) l'agent renonce
            journal.append("je n'avais plus rien à sacrifier ce jour-là")

        if impossible:
            return None, journal

    voyage = {"destination": demande["destination"], "date_depart": demande["date_depart"],
              "nuits": demande["nuits"], "voyageurs": demande["voyageurs"],
              "vol": vol, "hotel": hotels.iloc[ordre[position]], "activites": retenues,
              "prix_total": total, "budget_max": demande["budget_max"]}
    return voyage, journal


demande = {"destination": "Barcelone", "date_depart": "2026-08-12", "nuits": 4, "voyageurs": 2,
           "budget_max": 750, "envie": ENVIE}

voyage, journal = essayer_une_date(demande, conn, brochures, encodeur)
print()
afficher_voyage(voyage, journal)

gratuites = [a["nom"] for a in outil_activites(conn, "Barcelone").to_dict("records")
             if a["prix_eur"] == 0]
gardees = [a["nom"] for a in voyage["activites"]]

assert voyage is not None, "Un voyage devrait être trouvé pour cette demande"
assert voyage["prix_total"] <= demande["budget_max"], "Le voyage retenu doit tenir dans le budget"
assert all(nom in gardees for nom in gratuites), \
    "Retirer une sortie gratuite ne fait rien économiser : l'agent ne doit sacrifier que du payant"

# Une deuxième demande, pour voir le repli n°2 : un palace avec un budget trop juste.
palace = dict(demande, envie="le grand luxe, un palace avec spa", budget_max=950)
voyage_palace, journal_palace = essayer_une_date(palace, conn, brochures, encodeur)
print()
afficher_voyage(voyage_palace, journal_palace)

assert any("à la place" in ligne for ligne in journal_palace), \
    "Sur cette demande, l'agent doit finir par changer d'hôtel"
print("\nExercice validé !")

   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties  133 =    806 EUR   dépasse de 56
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   88 =    761 EUR   dépasse de 11
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   59 =    732 EUR   OK

=== Barcelone, 4 nuits, 2 voyageur(s) ===
  Vol    W67507 au départ de Nice le 2026-08-12 à 20:20   97 EUR/pers
  Hôtel  Casa Bellavista, 144 EUR/nuit : 576 EUR
  Activités :
      Plage de la Barceloneta                    0 EUR
      Marché de la Boqueria                      0 EUR
      Spectacle de la Fontaine magique de Montjuïc     0 EUR
      Musée Picasso                             15 EUR
      Parc Güell                                18 EUR
      Sagrada Familia                           26 EUR
  ----------------------------------------------------
  TOTAL  732 EUR   (budget : 750 EUR)

Ce que l'agent a fait, et ce qu'il a remarqué :
  - j'ai retiré « Dégustation de tapas dans le quartier d'El Bo

Améliorer l'Agent : chercher là où on ne lui a rien demandé

In [ ]:
def planifier(demande, conn, brochures, encodeur):
    """
    L'agent complet : il compose le voyage demandé, puis explore les jours voisins.

    Arguments :
    demande -- un dictionnaire avec destination, date_depart, nuits, voyageurs, budget_max, envie
    conn, brochures, encodeur -- les sources de données de l'agent

    Retourne :
    voyage -- le voyage pour la date demandée, ou None si rien n'y rentre
    journal -- les ajustements de l'agent, et le bon plan qu'il a repéré ailleurs
    """
    print(f"[agent] {demande['destination']}, {demande['nuits']} nuits, "
          f"{demande['voyageurs']} voyageur(s), budget {demande['budget_max']:.0f} EUR par personne")

    voyage, journal = essayer_une_date(demande, conn, brochures, encodeur)

    # On rejoue exactement le même raisonnement pour chaque jour voisin. Seul le prix du
    # vol change d'un jour à l'autre : les hôtels et les activités, eux, ne bougent pas.
    alternatives = []
    for autre_jour in dates_voisines(demande["date_depart"]):
        candidat, _ = essayer_une_date(dict(demande, date_depart=autre_jour), conn, brochures, encodeur)
        if candidat is not None:
            alternatives.append(candidat)

    if not alternatives:
        return voyage, journal

    if voyage is None:
        # Rien ne rentrait le jour demandé : on signale le jour voisin le moins cher.
        secours = min(alternatives, key=lambda v: v["prix_total"])
        jour = datetime.strptime(secours["date_depart"], "%Y-%m-%d").strftime("%d/%m")
        journal.append(f"en revanche, en partant le {jour}, un voyage à "
                       f"{secours['prix_total']:.0f} EUR par personne devenait possible")
        return voyage, journal

    meilleure = max(alternatives,key=lambda v: (len(v["activites"]), -v["prix_total"])) # (1) le meilleur jour voisin
    sorties_en_plus = len(meilleure["activites"]) - len(voyage["activites"]) # (2) les sorties gagnées
    economie = voyage["prix_total"] - meilleure["prix_total"] # (3) les euros gagnés

    jour = datetime.strptime(meilleure["date_depart"], "%Y-%m-%d").strftime("%d/%m")
    if sorties_en_plus > 0:
        journal.append(f"au passage, en partant le {jour} vous gardiez "
                       f"{len(meilleure['activites'])} sorties au lieu de {len(voyage['activites'])}, "
                       f"pour {meilleure['prix_total']:.0f} EUR")
    elif economie >= 10:
        journal.append(f"au passage, en partant le {jour} le même programme revenait à "
                       f"{meilleure['prix_total']:.0f} EUR, soit {economie:.0f} EUR de moins par personne")

    return voyage, journal


voyage, journal = planifier(demande, conn, brochures, encodeur)
print()
afficher_voyage(voyage, journal)

print()
print(texte_proposition(voyage, journal))

assert voyage is not None, "Un voyage devrait être trouvé pour cette demande"
assert voyage["date_depart"] == demande["date_depart"], \
    "L'agent ne doit jamais déplacer les dates tout seul : il signale, il ne décide pas"
assert any("en partant le" in ligne for ligne in journal), \
    "L'agent devrait avoir repéré un jour voisin moins cher et l'avoir noté dans le journal"
print("\nExercice validé !")

[agent] Barcelone, 4 nuits, 2 voyageur(s), budget 750 EUR par personne
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties  133 =    806 EUR   dépasse de 56
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   88 =    761 EUR   dépasse de 11
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   59 =    732 EUR   OK
   11/08  Casa Bellavista      vol   74 + hôtel   576 + sorties  133 =    783 EUR   dépasse de 33
   11/08  Casa Bellavista      vol   74 + hôtel   576 + sorties   88 =    738 EUR   OK
   13/08  Casa Bellavista      vol   83 + hôtel   576 + sorties  133 =    792 EUR   dépasse de 42
   13/08  Casa Bellavista      vol   83 + hôtel   576 + sorties   88 =    747 EUR   OK
   10/08  Casa Bellavista      vol   72 + hôtel   576 + sorties  133 =    781 EUR   dépasse de 31
   10/08  Casa Bellavista      vol   72 + hôtel   576 + sorties   88 =    736 EUR   OK
   14/08  Casa Bellavista      vol   79 + hôtel   576 + sorties  133 =    788 EUR   dépasse

In [ ]:
serre = dict(demande, budget_max=660)   # trop juste pour le 12 août, mais pas pour tous les jours
voyage_serre, journal_serre = planifier(serre, conn, brochures, encodeur)
print()
afficher_voyage(voyage_serre, journal_serre)

assert voyage_serre is None, "Avec ce budget, rien ne doit rentrer le jour demandé"
assert any("devenait possible" in ligne for ligne in journal_serre), \
    "L'agent devrait avoir trouvé un jour voisin où le voyage tient"
print("\nPersonne ne lui avait demandé de regarder les autres jours. C'est ça, un agent.")

[agent] Barcelone, 4 nuits, 2 voyageur(s), budget 660 EUR par personne
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties  133 =    806 EUR   dépasse de 146
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   88 =    761 EUR   dépasse de 101
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   59 =    732 EUR   dépasse de 72
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   33 =    706 EUR   dépasse de 46
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   15 =    688 EUR   dépasse de 28
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties    0 =    673 EUR   dépasse de 13
   11/08  Casa Bellavista      vol   74 + hôtel   576 + sorties  133 =    783 EUR   dépasse de 123
   11/08  Casa Bellavista      vol   74 + hôtel   576 + sorties   88 =    738 EUR   dépasse de 78
   11/08  Casa Bellavista      vol   74 + hôtel   576 + sorties   59 =    709 EUR   dépasse de 49
   11/08  Casa Bellavista      vol   74 + hô

In [ ]:
impossible = dict(demande, budget_max=200)
voyage_impossible, journal_impossible = planifier(impossible, conn, brochures, encodeur)
print()
afficher_voyage(voyage_impossible, journal_impossible)

assert voyage_impossible is None, "Avec 200 EUR, l'agent ne doit rien proposer du tout"
assert not any("devenait possible" in l for l in journal_impossible), \
    "Aucun jour ne tient dans 200 EUR : l'agent ne doit rien promettre"
print("\nUn agent qui invente une solution quand il n'y en a pas est un agent dangereux. Celui-ci sait dire non.")

[agent] Barcelone, 4 nuits, 2 voyageur(s), budget 200 EUR par personne
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties  133 =    806 EUR   dépasse de 606
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   88 =    761 EUR   dépasse de 561
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   59 =    732 EUR   dépasse de 532
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   33 =    706 EUR   dépasse de 506
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   15 =    688 EUR   dépasse de 488
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties    0 =    673 EUR   dépasse de 473
   11/08  Casa Bellavista      vol   74 + hôtel   576 + sorties  133 =    783 EUR   dépasse de 583
   11/08  Casa Bellavista      vol   74 + hôtel   576 + sorties   88 =    738 EUR   dépasse de 538
   11/08  Casa Bellavista      vol   74 + hôtel   576 + sorties   59 =    709 EUR   dépasse de 509
   11/08  Casa Bellavista      vol   7

Partie 5 - Laisser l'Agent effectuer une réservation à notre place

In [ ]:
from datetime import datetime


def reserver(conn, voyage, client, confirme=False):
    """
    Enregistre le voyage dans la table reservations, mais seulement si le voyageur a confirmé.

    Arguments :
    conn -- une connexion ouverte vers voyages.db
    voyage -- le dictionnaire retourné par planifier
    client -- le nom du voyageur
    confirme -- doit valoir True pour que quoi que ce soit soit écrit

    Retourne :
    message -- ce qui s'est passé, en clair
    """
    if voyage is None:
        return "Rien à réserver : aucun voyage n'a été trouvé."

    # LE GARDE-FOU. Trois lignes, et c'est la partie la plus importante du fichier :
    # tant que le voyageur n'a pas dit oui, la fonction ressort sans rien avoir écrit.
    if not confirme:
        return (f"Rien n'a été réservé. Le voyage à {voyage['destination']} coûterait "
                f"{voyage['prix_total']:.0f} EUR par personne, soit "
                f"{voyage['prix_total'] * voyage['voyageurs']:.0f} EUR au total. "
                f"Il faut confirmer pour que la réservation soit enregistrée.")

    ### START CODE HERE ###

    sql = """
        INSERT INTO reservations (client, destination, date_depart, nuits, voyageurs, vol, hotel,
                                  activites, prix_total, reservee_le)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """  # (1) dix colonnes, donc dix points d'interrogation
    conn.execute(sql, (client, voyage["destination"], voyage["date_depart"], voyage["nuits"],
                       voyage["voyageurs"], voyage["vol"]["numero"], voyage["hotel"]["hotel"],
                       ", ".join(a["nom"] for a in voyage["activites"]),
                       voyage["prix_total"] * voyage["voyageurs"],
                       datetime.now().strftime("%Y-%m-%d %H:%M")))

    ### START CODE HERE ###

    conn.commit() # (2) enregistre le résultat dans la base de données
    return (f"C'est réservé pour {client} : {voyage['destination']}, "
            f"{voyage['nuits']} nuits, {voyage['prix_total'] * voyage['voyageurs']:.0f} EUR au total.")


def combien_de_reservations():
    return len(requete(conn, "SELECT * FROM reservations"))


avant = combien_de_reservations()

print(reserver(conn, voyage, "Marc"))                      # sans confirmation
apres_refus = combien_de_reservations()

print()
print(reserver(conn, voyage, "Marc", confirme=True))       # avec confirmation
apres_accord = combien_de_reservations()

display(requete(conn, "SELECT client, destination, hotel, prix_total, reservee_le FROM reservations"))

assert apres_refus == avant, "Sans confirmation, l'agent ne doit RIEN ecrire dans la base"
assert apres_accord == avant + 1, "Avec confirmation, une ligne doit etre ajoutee"
print("Exercice validé ! Le premier appel n'a rien écrit, le second oui.")

Rien n'a été réservé. Le voyage à Barcelone coûterait 732 EUR par personne, soit 1464 EUR au total. Il faut confirmer pour que la réservation soit enregistrée.

C'est réservé pour Marc : Barcelone, 4 nuits, 1464 EUR au total.


,client,destination,hotel,prix_total,reservee_le
0,Marc,Barcelone,Casa Bellavista,1464.0,2026-08-13 14:13
1,Marc,Barcelone,Casa Bellavista,1464.0,2026-11-17 12:28


Exercice validé ! Le premier appel n'a rien écrit, le second oui.


Partie 6 - La cerise sur le gâteau : l'application Web

In [23]:
import importlib
import inspect
from pathlib import Path

EN_TETE = (
    "# L'agent de voyage, écrit automatiquement depuis le notebook du Projet 07.\n"
    "from datetime import datetime, timedelta\n\n"
    "from utils import encoder, requete\n\n\n"
)

fonctions = [outil_vols, outil_activites, outil_hotels, chiffrer_voyage,
             dates_voisines, essayer_une_date, planifier, reserver]
corps = "\n\n".join(inspect.getsource(f) for f in fonctions)

Path("agent.py").write_text(EN_TETE + corps, encoding="utf-8")
print(f"agent.py écrit : {len(fonctions)} fonctions, {len((EN_TETE + corps).splitlines())} lignes")

import agent
agent = importlib.reload(agent)   # au cas où tu réexécutes cette cellule après une modification
assert hasattr(agent, "planifier"), "agent.py doit contenir la fonction planifier"
assert hasattr(agent, "reserver"), "agent.py doit contenir la fonction reserver"
print("Import vérifié !")

agent.py écrit : 8 fonctions, 277 lignes
Import vérifié !


Étape Finale : branche ton agent dans l'application

In [24]:
import datetime as dt

# Les variables, telles que app.py les reçoit de son formulaire
destination = "Barcelone"
depart = dt.date(2026, 8, 12)
nuits = 4
voyageurs = 2
budget = 750
envie = ENVIE

### START CODE HERE ###

date_texte = depart.strftime("%Y-%m-%d")
demande = {
    "destination": destination,
    "date_depart": date_texte,
    "nuits": nuits,
    "voyageurs": voyageurs,
    "budget_max": budget,
    "envie": envie
} # (2) la demande complète, avec ses 6 clés
voyage, journal = agent.planifier(demande, conn, brochures, encodeur) # (3) et on appelle l'agent

### END CODE HERE ###

print(f"Voyage trouvé : {voyage['hotel']['hotel']}, {voyage['prix_total']:.0f} EUR par personne")

assert demande["date_depart"] == "2026-08-12", "La date doit être convertie en texte, au format AAAA-MM-JJ"
assert set(demande) == {"destination", "date_depart", "nuits", "voyageurs", "budget_max", "envie"}, \
    "La demande doit contenir exactement les 6 clés attendues par l'agent"
assert isinstance(journal, list), "planifier retourne deux choses : le voyage, puis le journal (une liste)"
assert voyage is not None and voyage["prix_total"] <= budget, "Le voyage doit tenir dans le budget"
print("Exercice validé ! Recopie maintenant ces trois lignes dans app.py, à la place des trois None.")

[agent] Barcelone, 4 nuits, 2 voyageur(s), budget 750 EUR par personne
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties  133 =    806 EUR   dépasse de 56
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   88 =    761 EUR   dépasse de 11
   12/08  Casa Bellavista      vol   97 + hôtel   576 + sorties   59 =    732 EUR   OK
   11/08  Casa Bellavista      vol   74 + hôtel   576 + sorties  133 =    783 EUR   dépasse de 33
   11/08  Casa Bellavista      vol   74 + hôtel   576 + sorties   88 =    738 EUR   OK
   13/08  Casa Bellavista      vol   83 + hôtel   576 + sorties  133 =    792 EUR   dépasse de 42
   13/08  Casa Bellavista      vol   83 + hôtel   576 + sorties   88 =    747 EUR   OK
   10/08  Casa Bellavista      vol   72 + hôtel   576 + sorties  133 =    781 EUR   dépasse de 31
   10/08  Casa Bellavista      vol   72 + hôtel   576 + sorties   88 =    736 EUR   OK
   14/08  Casa Bellavista      vol   79 + hôtel   576 + sorties  133 =    788 EUR   dépasse